In [23]:
import cv2
import json
import os
import glob

# --- CONFIG ---
DATASET_BASE = "D:/GitHub/BaseballPitch/modules/pitcher_detection/curated_finetuning_dataset"
SPLIT = "val"  # "train" or "val" or "test"
PITCH_TYPE = "SL - Slider"
# Progress tracking file — records which reviews are completed
REVIEW_PROGRESS_FILE = f"{DATASET_BASE}/review_progress.json"
# --------------

CLASS_NAMES = ['pitcher']
CLASS_COLORS = [(0, 255, 0)]  # Green for pitcher

In [ ]:
def _load_review_progress():
    if os.path.exists(REVIEW_PROGRESS_FILE):
        with open(REVIEW_PROGRESS_FILE, "r") as f:
            return json.load(f)
    return {}

def _save_review_progress(progress):
    os.makedirs(os.path.dirname(REVIEW_PROGRESS_FILE), exist_ok=True)
    with open(REVIEW_PROGRESS_FILE, "w") as f:
        json.dump(progress, f, indent=2)

def _review_progress_key():
    return f"review/{SPLIT}/{PITCH_TYPE}"

def draw_yolo_box(img, label_line):
    """Draw bounding box and pose keypoints from YOLO format label"""
    h, w = img.shape[:2]
    label_line = label_line.replace('\\n', '').strip()
    parts = label_line.strip().split()
    if len(parts) < 5:
        return
    
    # YOLO format: class x_center y_center width height [kpts...]
    cls = int(parts[0])
    x_c, y_c, box_w, box_h = map(float, parts[1:5])
    
    # Convert box to pixel coordinates
    x_c *= w
    y_c *= h
    box_w *= w
    box_h *= h
    
    x1 = int(x_c - box_w/2)
    y1 = int(y_c - box_h/2)
    x2 = int(x_c + box_w/2)
    y2 = int(y_c + box_h/2)
    
    # Get class info
    class_name = CLASS_NAMES[cls] if cls < len(CLASS_NAMES) else f"Class {cls}"
    color = CLASS_COLORS[cls] if cls < len(CLASS_COLORS) else (255, 255, 255)
    
    # Draw bounding box
    cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
    
    # Draw label background
    label_text = class_name
    (text_w, text_h), _ = cv2.getTextSize(label_text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
    cv2.rectangle(img, (x1, y1 - text_h - 10), (x1 + text_w + 10, y1), color, -1)
    cv2.putText(img, label_text, (x1 + 5, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    # Draw pose keypoints if present (x, y, v triplets)
    if len(parts) > 5:
        kpt_values = parts[5:]
        kpt_count = len(kpt_values) // 3
        for i in range(kpt_count):
            kx = float(kpt_values[i * 3]) * w
            ky = float(kpt_values[i * 3 + 1]) * h
            vis = float(kpt_values[i * 3 + 2])
            if vis > 0:
                cv2.circle(img, (int(kx), int(ky)), 3, (0, 0, 255), -1)

def _video_prefix_from_image_name(img_name):
    stem = os.path.splitext(img_name)[0]
    return stem.rsplit('_', 1)[0] if '_' in stem else stem

def review_labels():
    progress = _load_review_progress()
    progress_key = _review_progress_key()

    # Load set of already-reviewed filenames
    entry = progress.get(progress_key, {})
    reviewed_files = set(entry.get("reviewed", []))

    img_dir = os.path.join(DATASET_BASE, "images", SPLIT, PITCH_TYPE)
    label_dir = os.path.join(DATASET_BASE, "labels", SPLIT, PITCH_TYPE)
    
    # Get all images, then filter to only unreviewed
    all_images = sorted(glob.glob(os.path.join(img_dir, "*.jpg")))
    images = [p for p in all_images if os.path.basename(p) not in reviewed_files]
    
    if not images:
        print(f"⏭️  All {len(all_images)} images already reviewed in {SPLIT}/{PITCH_TYPE}")
        print(f"   (Reviewed: {len(reviewed_files)}, Deleted so far: {entry.get('deleted', 0)})")
        return
    
    print(f"Found {len(images)} new images to review in {SPLIT}/{PITCH_TYPE}")
    if reviewed_files:
        print(f"  (Already reviewed: {len(reviewed_files)}, previously deleted: {entry.get('deleted', 0)})")
    print("\nControls:")
    print("  SPACE/RIGHT ARROW/S - Next image")
    print("  LEFT ARROW/A - Previous image")
    print("  D - Delete current image and label")
    print("  P - Delete ALL images+labels for this video")
    print("  Q/ESC - Quit")
    print("\n" + "="*50)
    
    idx = 0
    deleted_count = 0
    newly_reviewed = []
    
    while idx < len(images):
        img_path = images[idx]
        img_name = os.path.basename(img_path)
        label_path = os.path.join(label_dir, img_name.replace('.jpg', '.txt'))
        
        # Load image
        img = cv2.imread(img_path)
        if img is None:
            print(f"Failed to load {img_path}")
            idx += 1
            continue
        
        # Draw bounding boxes if label exists
        has_label = False
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                lines = f.readlines()
                for line in lines:
                    draw_yolo_box(img, line)
                has_label = True
        
        # Add info overlay
        info_text = f"[{idx+1}/{len(images)}] {img_name}"
        if not has_label:
            info_text += " (NO LABEL)"
        cv2.putText(img, info_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 255, 255), 2)
        cv2.putText(img, info_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 1)
        
        # Show image
        cv2.imshow("Label Review", img)
        
        # Wait for key (use waitKeyEx for reliable arrow keys on Windows)
        key = cv2.waitKeyEx(0)

        # Quit (partial — save what we've reviewed so far)
        if key in (ord('q'), ord('Q'), 27):  # Q or ESC
            break
        # Delete current image and label
        elif key in (ord('d'), ord('D')):  # Delete one
            os.remove(img_path)
            if os.path.exists(label_path):
                os.remove(label_path)
            # Track as reviewed (deleted files won't reappear anyway)
            newly_reviewed.append(img_name)
            images.pop(idx)
            deleted_count += 1
            print(f"Deleted: {img_name}")
            idx = max(0, idx)  # Stay at same position
        # Delete all images/labels for current video prefix
        elif key in (ord('p'), ord('P')):
            video_prefix = _video_prefix_from_image_name(img_name)
            video_images = sorted(glob.glob(os.path.join(img_dir, f"{video_prefix}_*.jpg")))
            if not video_images:
                # Fallback in case naming doesn't include frame suffix.
                candidate = os.path.join(img_dir, f"{video_prefix}.jpg")
                if os.path.exists(candidate):
                    video_images = [candidate]

            deleted_this_video = 0
            removed_basenames = set()
            for vp in video_images:
                base = os.path.basename(vp)
                vp_label = os.path.join(label_dir, base.replace('.jpg', '.txt'))
                try:
                    if os.path.exists(vp):
                        os.remove(vp)
                    if os.path.exists(vp_label):
                        os.remove(vp_label)
                    deleted_this_video += 1
                    removed_basenames.add(base)
                except Exception as ex:
                    print(f"Failed deleting {base}: {ex}")

            if deleted_this_video > 0:
                deleted_count += deleted_this_video
                print(f"Deleted {deleted_this_video} frames for video prefix: {video_prefix}")
            else:
                print(f"No files found for video prefix: {video_prefix}")

            # Remove deleted video files from current queue and progress tracking in memory.
            if removed_basenames:
                images = [p for p in images if os.path.basename(p) not in removed_basenames]
                reviewed_files.difference_update(removed_basenames)
                newly_reviewed = [n for n in newly_reviewed if n not in removed_basenames]

            if not images:
                break
            idx = min(idx, len(images) - 1)
        # Next image
        elif key in (
            32,               # Space
            2555904,          # Right arrow (Windows)
            83,               # Right arrow (Linux/X11)
            ord('s'), ord('S')  # 'S' key
        ):
            # Mark current image as reviewed when moving past it
            newly_reviewed.append(img_name)
            idx = min(idx + 1, len(images) - 1)
        # Previous image
        elif key in (
            2424832,          # Left arrow (Windows)
            81,               # Left arrow (Linux/X11)
            8,                # Backspace
            ord('a'), ord('A')  # 'A' key
        ):
            idx = max(idx - 1, 0)
    
    cv2.destroyAllWindows()

    # Merge newly reviewed into progress
    reviewed_files.update(newly_reviewed)
    prior_deleted = entry.get("deleted", 0)

    progress[progress_key] = {
        "reviewed": sorted(reviewed_files),
        "deleted": prior_deleted + deleted_count,
    }
    _save_review_progress(progress)

    print(f"\nReview session complete!")
    print(f"  Newly reviewed: {len(newly_reviewed)}")
    print(f"  Deleted this session: {deleted_count}")
    print(f"  Total reviewed: {len(reviewed_files)}")
    print(f"  Total deleted:  {prior_deleted + deleted_count}")

In [25]:
review_labels()

Found 730 new images to review in val/SL - Slider

Controls:
  SPACE/RIGHT ARROW/S - Next image
  LEFT ARROW/A - Previous image
  D - Delete current image and label
  Q/ESC - Quit


Review session complete!
  Newly reviewed: 1461
  Deleted this session: 0
  Total reviewed: 730
  Total deleted:  0


In [10]:
# --- Review Progress Summary ---
progress = _load_review_progress()

review_entries = {k: v for k, v in progress.items() if k.startswith("review/")}

if not review_entries:
    print("No review progress recorded yet.")
else:
    total_reviewed = sum(len(v.get("reviewed", [])) for v in review_entries.values())
    total_deleted = sum(v.get("deleted", 0) for v in review_entries.values())

    print(f"{'='*60}")
    print(f"Review Progress Summary")
    print(f"{'='*60}")
    print(f"  Total reviewed: {total_reviewed}")
    print(f"  Total deleted:  {total_deleted}")
    print(f"{'='*60}")

    for key in sorted(review_entries):
        entry = review_entries[key]
        reviewed = len(entry.get("reviewed", []))
        deleted = entry.get("deleted", 0)
        parts = key.split("/", 2)  # review / split / pitch_type
        split_name = parts[1] if len(parts) > 1 else "?"
        pitch_name = parts[2] if len(parts) > 2 else "?"
        print(f"  [{split_name}] {pitch_name}  —  "
              f"reviewed: {reviewed}, deleted: {deleted}")

Review Progress Summary
  Total reviewed: 1800
  Total deleted:  21
  [val] CH - Changeup  —  reviewed: 900, deleted: 21
  [val] ST - Sweeper  —  reviewed: 900, deleted: 0


In [29]:
# --- Delete by Play ID prefix (safe utility) ---
from pathlib import Path
import re

# Required: play id or prefix. If you pass a full play id, only the first 8 chars are used.
TARGET_PLAY_ID = "d4a9f0bd"  # e.g. "89647140-7238-3a08-a3ae-76c52a78c501"
PREFIX_LEN = 8

# Scope controls
SEARCH_ALL_SPLITS_AND_PITCH_TYPES = False  # True = entire curated_finetuning_dataset, False = current SPLIT/PITCH_TYPE only

# Safety controls
DRY_RUN = False   # True = preview only, False = actually delete
CONFIRM_DELETE = "DELETE"  # set to "DELETE" when DRY_RUN=False

def _build_search_roots(dataset_base, split, pitch_type, search_all):
    base = Path(dataset_base)
    if search_all:
        return [base / "images", base / "labels"]
    return [
        base / "images" / split / pitch_type,
        base / "labels" / split / pitch_type,
    ]

def _matches_play_prefix(file_path, prefix):
    stem = file_path.stem  # filename without extension
    stem_lower = stem.lower()
    prefix_lower = prefix.lower()

    # Primary: match the PlayID token if present (e.g., PlayID-89647140-xxxx...).
    m = re.search(r"playid-([0-9a-fA-F\-]+)", stem, flags=re.IGNORECASE)
    if m:
        play_id_token = m.group(1).lower()
        if play_id_token.startswith(prefix_lower):
            return True

    # Fallback: generic contains/startswith match in case of different filename format.
    return (prefix_lower in stem_lower) or stem_lower.startswith(prefix_lower)

def delete_by_play_prefix(
    dataset_base,
    target_play_id,
    prefix_len=8,
    search_all=True,
    dry_run=True,
    confirm_delete="",
):
    target_play_id = str(target_play_id).strip()
    if not target_play_id:
        print("Provide TARGET_PLAY_ID before running this cell.")
        return

    prefix = target_play_id[:prefix_len]
    if not prefix:
        print("Computed prefix is empty. Check TARGET_PLAY_ID / PREFIX_LEN.")
        return

    roots = _build_search_roots(dataset_base, SPLIT, PITCH_TYPE, search_all)
    image_exts = {".jpg", ".jpeg", ".png"}
    label_exts = {".txt"}

    image_matches = []
    label_matches = []

    for root in roots:
        if not root.exists():
            continue
        for p in root.rglob("*"):
            if not p.is_file():
                continue
            ext = p.suffix.lower()
            if ext not in image_exts and ext not in label_exts:
                continue
            if _matches_play_prefix(p, prefix):
                if ext in image_exts:
                    image_matches.append(p)
                else:
                    label_matches.append(p)

    image_matches = sorted(set(image_matches))
    label_matches = sorted(set(label_matches))

    print(f"Play prefix: {prefix}")
    print(f"Images matched: {len(image_matches)}")
    print(f"Labels matched: {len(label_matches)}")

    # Show a short preview so you can verify before deletion.
    preview = (image_matches + label_matches)[:20]
    if preview:
        print("\nPreview (up to 20 paths):")
        for p in preview:
            print(f"  {p}")

    if dry_run:
        print("\nDRY_RUN=True, nothing deleted.")
        return

    if confirm_delete != "DELETE":
        print("\nDeletion blocked. Set CONFIRM_DELETE = 'DELETE' and rerun.")
        return

    deleted = 0
    failed = 0
    for p in image_matches + label_matches:
        try:
            p.unlink()
            deleted += 1
        except Exception as ex:
            failed += 1
            print(f"Failed to delete {p}: {ex}")

    print(f"\nDeleted files: {deleted}")
    print(f"Failed deletions: {failed}")

# Run
delete_by_play_prefix(
    dataset_base=DATASET_BASE,
    target_play_id=TARGET_PLAY_ID,
    prefix_len=PREFIX_LEN,
    search_all=SEARCH_ALL_SPLITS_AND_PITCH_TYPES,
    dry_run=DRY_RUN,
    confirm_delete=CONFIRM_DELETE,
)

Play prefix: d4a9f0bd
Images matched: 180
Labels matched: 180

Preview (up to 20 paths):
  D:\GitHub\BaseballPitch\modules\pitcher_detection\curated_finetuning_dataset\images\val\SL - Slider\PitchType-SL_Zone-13_PlayID-d4a9f0bd-3d8e-385b-a24d-f1b38eba4c14_Date-2025-07-10_0000.jpg
  D:\GitHub\BaseballPitch\modules\pitcher_detection\curated_finetuning_dataset\images\val\SL - Slider\PitchType-SL_Zone-13_PlayID-d4a9f0bd-3d8e-385b-a24d-f1b38eba4c14_Date-2025-07-10_0001.jpg
  D:\GitHub\BaseballPitch\modules\pitcher_detection\curated_finetuning_dataset\images\val\SL - Slider\PitchType-SL_Zone-13_PlayID-d4a9f0bd-3d8e-385b-a24d-f1b38eba4c14_Date-2025-07-10_0002.jpg
  D:\GitHub\BaseballPitch\modules\pitcher_detection\curated_finetuning_dataset\images\val\SL - Slider\PitchType-SL_Zone-13_PlayID-d4a9f0bd-3d8e-385b-a24d-f1b38eba4c14_Date-2025-07-10_0003.jpg
  D:\GitHub\BaseballPitch\modules\pitcher_detection\curated_finetuning_dataset\images\val\SL - Slider\PitchType-SL_Zone-13_PlayID-d4a9f0bd-3d8e